# Controlling the OpenMM integrator live

This notebook simulates a graphene sheet with OpenMM, exposing the integrator parameters to real time control from both the notebook and the headset.

## OpenMM simulation setup

We'll load a pre-bundled OpenMM simulation for convenience:

In [1]:
from nanover.openmm import unbundle_openmm_simulation

simulation = unbundle_openmm_simulation("../systems/openmm/graphene_restrained.openmm.zip")

## NanoVer simulation and server setup

Next we wrap the OpenMM simulation and server it with NanoVer, in the usual way:

In [2]:
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_simulation(simulation)

In [3]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(omm_sim, port=0, name="openmm graphene example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "openmm graphene example" (ws://localhost:51017), discoverable on all interfaces on port 54545
Available simulations:
[0]: "Unnamed OpenMM Simulation"
Switched to [0]: "Unnamed OpenMM Simulation"
Switched to [0]: "Unnamed OpenMM Simulation"


## Controlling the integrator from the notebook

This system uses an integrator ([LangevinIntegrator](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.LangevinIntegrator.html)) that supports changing some parameters at runtime. We can easily create some sliders to change temperature, friction, and timestep on the fly:

In [4]:
def set_temperature(temperature=300):
    omm_sim.simulation.integrator.setTemperature(temperature)

def set_friction(friction=1):
    omm_sim.simulation.integrator.setFriction(friction)

def set_timestep(timestep=0.5):
    omm_sim.simulation.integrator.setStepSize(timestep / 1000.0)

In [5]:
from ipywidgets import interact

temperature_min_max = 0, 10000
friction_min_max_step = 0.01, 100, 1.0
timestep_min_max_step = 0.01, 1.5, 0.01

interact(set_temperature, temperature=temperature_min_max)
interact(set_friction, friction=friction_min_max_step)
interact(set_timestep, timestep=timestep_min_max_step);

interactive(children=(IntSlider(value=300, description='temperature', max=10000), Output()), _dom_classes=('wi…

interactive(children=(FloatSlider(value=1.0, description='friction', min=0.01, step=1.0), Output()), _dom_clas…

interactive(children=(FloatSlider(value=0.5, description='timestep', max=1.5, min=0.01, step=0.01), Output()),…

## Controlling the integrator from the headset

We can also use the experimental panel system to show controls inside the XR headset for changing these parameters:

In [6]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [18]:
def set_integrator_params(temperature, friction, timestep):
    set_temperature(temperature)
    set_friction(friction)
    set_timestep(timestep)

utilities.define_command("integrator/set-params", handler=set_integrator_params)

In [20]:
utilities.panels.update_panel(
    "test",
    utilities.panels.slider(label="Temperature (K)", range=temperature_min_max, variable="variable.temperature"),
    utilities.panels.slider(label="Friction (1/fs)", range=friction_min_max_step[:2], variable="variable.friction"),
    utilities.panels.slider(label="Timestep (fs)", range=timestep_min_max_step[:2], variable="variable.timestep"),
    utilities.panels.button(
        label=f"Apply",
        command="integrator/set-params",
        arguments=dict(
            temperature=dict(variable="variable.temperature"),
            friction=dict(variable="variable.friction"),
            timestep=dict(variable="variable.timestep"),
        ),
    ),
    label="Integrator settings",
)